## Importing Libraries

In [1]:
import base64
import io
from pathlib import Path
from PIL import Image

# LangChain Imports
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama, OllamaEmbeddings

# Docling Imports
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.document import TextItem, TableItem
from docling.chunking import HybridChunker

/Users/animesh/anaconda3/envs/agentic-rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading Models

In [2]:
text_llm = ChatOllama(
    model="qwen3:8b",
    temperature=0
)

vision_llm = ChatOllama(
    model="qwen3.5:4b",
    temperature=0
)

embeddings = OllamaEmbeddings(
    model="bge-m3"
)

## Docling Configuration

In [3]:
pipeline_options = PdfPipelineOptions()
pipeline_options.generate_page_images = True
pipeline_options.generate_picture_images = True
pipeline_options.do_ocr = True

# Ensure table structure model is enabled
pipeline_options.do_table_structure = True

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

pdf_file = "./2410.10934v2.pdf"
output_dir = Path("./extracted_visuals")
output_dir.mkdir(parents=True, exist_ok=True)

print("Running Docling layout and table analysis...")
conv_res = converter.convert(pdf_file)
doc = conv_res.document

Running Docling layout and table analysis...


[INFO] 2026-10-12 14:31:59,272 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-10-12 14:31:59,282 [RapidOCR] download_file.py:60: File exists and is valid: /Users/animesh/anaconda3/envs/agentic-rag/lib/python3.11/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-10-12 14:31:59,282 [RapidOCR] main.py:63: Using /Users/animesh/anaconda3/envs/agentic-rag/lib/python3.11/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-10-12 14:31:59,310 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-10-12 14:31:59,312 [RapidOCR] download_file.py:60: File exists and is valid: /Users/animesh/anaconda3/envs/agentic-rag/lib/python3.11/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-10-12 14:31:59,313 [RapidOCR] main.py:63: Using /Users/animesh/anaconda3/envs/agentic-rag/lib/python3.11/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-10-12 14:31:59,334 [RapidOCR] base.py:23: Using en

## CHUNK TABLE

In [4]:
import pandas as pd
from langchain_core.documents import Document

def chunk_table(
    table_item,
    table_index: int,
    max_rows_per_chunk: int = 15
) -> list[Document]:
    """
    Chunks a Docling table into row batches while preserving column headers
    in every markdown segment.
    """
    df: pd.DataFrame = table_item.export_to_dataframe()
    
    # Guard against empty/malformed tables
    if df.empty:
        return []

    total_rows = len(df)
    page_no = table_item.prov[0].page_no if table_item.prov else None
    columns = [str(c) for c in df.columns]

    documents = []

    # Iterate in row slices (single-iteration if total_rows <= max_rows_per_chunk)
    for start_idx in range(0, total_rows, max_rows_per_chunk):
        end_idx = min(start_idx + max_rows_per_chunk, total_rows)
        sub_df = df.iloc[start_idx:end_idx]

        # Markdown conversion retains the header row automatically
        table_markdown = sub_df.to_markdown(index=False)

        # Append row slice indicator only if the table was actually split
        slice_tag = f" [Rows {start_idx + 1}–{end_idx} of {total_rows}]" if total_rows > max_rows_per_chunk else ""
        
        content = (
            f"### Table {table_index + 1} (Page {page_no}){slice_tag}\n\n"
            f"{table_markdown}"
        )

        documents.append(
            Document(
                page_content=content,
                metadata={
                    "chunk_id": f"table_{table_index + 1}_rows_{start_idx + 1}_{end_idx}",
                    "type": "table",
                    "page_number": page_no,
                    "rows_range": f"{start_idx + 1}-{end_idx}",
                    "total_rows": total_rows,
                    "columns": columns,
                }
            )
        )

    return documents

In [5]:
langchain_documents: list[Document] = []

for idx, table in enumerate(doc.tables):
    # Chunks tables slices with preserved headers
    table_chunks = chunk_table(
        table_item=table,
        table_index=idx,
        max_rows_per_chunk=10,  # rows of table per chunk
    )
    langchain_documents.extend(table_chunks)

print(f"Total Table Chunks Created: {sum(1 for d in langchain_documents if d.metadata['type'] == 'table')}")

Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


Total Table Chunks Created: 10


In [6]:
langchain_documents

[Document(metadata={'chunk_id': 'table_1_rows_1_9', 'type': 'table', 'page_number': 6, 'rows_range': '1-9', 'total_rows': 9, 'columns': ['Metric', 'MetaGPT (Hong et al., 2024b)', 'GPT-Pilot (Pythagora.io, 2023)', 'OpenHands (Wang et al., 2024d)']}, page_content='### Table 1 (Page 6)\n\n| Metric                       | MetaGPT (Hong et al., 2024b)          | GPT-Pilot (Pythagora.io, 2023)   | OpenHands (Wang et al., 2024d)    |\n|:-----------------------------|:--------------------------------------|:---------------------------------|:----------------------------------|\n| Basic Statistics             | Basic Statistics                      | Basic Statistics                 | Basic Statistics                  |\n| Version                      | Data Interpreter (Hong et al., 2024a) | 0.2.13                           | CodeAct v1.9 (Wang et al., 2024c) |\n| (1) Average Cost             | $ 1.19                                | $ 3.92                           | $ 6.38                   

## Chart Transcription

In [7]:
def get_surrounding_context(target_item, document, window_size=2) -> str:
    body_items = list(document.iterate_items())
    target_idx = -1
    for idx, (item, _) in enumerate(body_items):
        if item == target_item:
            target_idx = idx
            break

    if target_idx == -1:
        return "No adjacent context available."

    start_idx = max(0, target_idx - window_size)
    end_idx = min(len(body_items), target_idx + window_size + 1)

    context_snippets = []
    for i in range(start_idx, end_idx):
        if i == target_idx:
            continue
        curr_item, _ = body_items[i]
        if isinstance(curr_item, TextItem) and curr_item.text.strip():
            context_snippets.append(curr_item.text.strip())

    return "\n".join(context_snippets)

## RESIZE and TRANSCRIBE
def process_and_resize_image(pil_img, target_size: tuple[int, int] = (300, 200)) -> str:
    """
    Converts image to RGB, resizes it to target dimensions (e.g., 200x200),
    and returns the base64 data string.
    """
        
    # Resize with high-quality antialiasing
    resized_img = pil_img.resize(target_size, Image.Resampling.LANCZOS)
    
    # Encode directly to base64 via memory buffer
    buffer = io.BytesIO()
    resized_img.save(buffer, format="JPEG", quality=85)
    return base64.b64encode(buffer.getvalue()).decode("utf-8")


def transcribe_chart_with_langchain(pil_image, surrounding_context: str) -> str:

    base64_image = process_and_resize_image(pil_image, target_size=(300, 200))
    prompt_text = f"""
You are an expert technical and financial document analyst.
Analyze this cropped figure/chart from an enterprise document.

SURROUNDING DOCUMENT TEXT:
---
{surrounding_context}
---

INSTRUCTIONS:
1. Ground your interpretation in the surrounding text.
2. Transcribe axes, legends, units, and data points into a clean Markdown table.
3. Provide a concise 2-sentence summary of the main insight.
"""
    message = HumanMessage(
        content=[
            {"type": "text", "text": prompt_text},
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{base64_image}"},
            },
        ]
    )
    return vision_llm.invoke([message]).content

### Filter Pictures

In [8]:
from PIL import Image
from pathlib import Path


def get_picture_metrics(picture, doc):
    """Extract image dimensions and page-relative bounding-box area."""
    image = picture.get_image(doc)

    if image is None:
        return None

    width, height = image.size

    metrics = {
        "image": image,
        "width": width,
        "height": height,
        "pixel_area": width * height,
        "page_area_ratio": None,
        "aspect_ratio": width / max(height, 1),
    }

    # Try to calculate the picture's area relative to its PDF page.
    try:
        if picture.prov:
            prov = picture.prov[0]
            page = doc.pages[prov.page_no]
            bbox = prov.bbox

            page_width = page.size.width
            page_height = page.size.height

            bbox_width = abs(bbox.r - bbox.l)
            bbox_height = abs(bbox.t - bbox.b)

            page_area = page_width * page_height

            if page_area > 0:
                metrics["page_area_ratio"] = (
                    bbox_width * bbox_height / page_area
                )
    except (AttributeError, KeyError, IndexError, TypeError):
        # Pixel-based filtering can still be used.
        pass

    return metrics


def is_relevant_picture(
    picture,
    doc,
    min_width=100,
    min_height=80,
    min_pixel_area=12_000,
    min_page_area_ratio=0.002,
):
    """
    Heuristic filter: reject very small images and tiny page regions.

    Tune the thresholds against your own PDFs.
    """
    metrics = get_picture_metrics(picture, doc)

    if metrics is None:
        return False, "no_image", None

    width = metrics["width"]
    height = metrics["height"]
    page_ratio = metrics["page_area_ratio"]

    if width < min_width or height < min_height:
        return False, "small_dimensions", metrics

    if metrics["pixel_area"] < min_pixel_area:
        return False, "small_pixel_area", metrics

    if (
        page_ratio is not None
        and page_ratio < min_page_area_ratio
    ):
        return False, "small_page_area", metrics

    return True, "passed_heuristics", metrics

In [ ]:
from collections import Counter

output_dir = Path("./extracted_visuals")
output_dir.mkdir(parents=True, exist_ok=True)

filter_stats = Counter()
visual_documents = []

print(f"Found {len(doc.pictures)} picture candidates.")

for idx, picture in enumerate(doc.pictures):
    keep, reason, metrics = is_relevant_picture(picture, doc)
    filter_stats[reason] += 1

    if not keep:
        print(f"Skipping picture {idx}: {reason}")
        continue

    pil_image = metrics["image"]
    page_no = picture.prov[0].page_no if picture.prov else None

    img_path = output_dir / f"figure_{idx + 1}.png"
    pil_image.save(img_path)

    # Only call the expensive vision model for candidates
    # that have passed the initial filtering.
    context_text = get_surrounding_context(
        picture, doc, window_size=2
    )

    transcription = transcribe_chart_with_langchain(
        pil_image, context_text
    )

    visual_documents.append(
        Document(
            page_content=(
                f"### Visual Data / Figure\n"
                f"{transcription}\n\n"
                f"**Surrounding Context:**\n{context_text}"
            ),
            metadata={
                "chunk_id": f"visual_{idx + 1}",
                "type": "chart_or_figure",
                "page_number": page_no,
                "image_path": str(img_path),
                "image_width": metrics["width"],
                "image_height": metrics["height"],
                "page_area_ratio": metrics["page_area_ratio"],
            },
        )
    )

langchain_documents.extend(visual_documents)

print("\nFilter summary:")
print(dict(filter_stats))
print(f"Relevant visual chunks: {len(visual_documents)}")

## Docling Hybrid Text Chunking

In [ ]:
chunker = HybridChunker(
    tokenizer="sentence-transformers/all-MiniLM-L6-v2",
    max_tokens=512,
    merge_peers=True,
)

doc_chunks = list(chunker.chunk(doc))

for idx, chunk in enumerate(doc_chunks):
    # Skip chunk if it only contains table items (since we explicitly handled tables above)
    has_only_tables = chunk.meta.doc_items and all(
        isinstance(item, TableItem) for item in chunk.meta.doc_items
    )
    if has_only_tables:
        continue

    page_no = chunk.meta.doc_items[0].prov[0].page_no if chunk.meta.doc_items else None
    langchain_documents.append(
        Document(
            page_content=chunker.contextualize(chunk=chunk),
            metadata={
                "chunk_id": f"text_chunk_{idx}",
                "type": "text",
                "page_number": page_no,
                "image_path": None,
            },
        )
    )

print(f"Total LangChain documents indexed: {len(langchain_documents)}")
print(f"- Tables: {sum(1 for d in langchain_documents if d.metadata['type'] == 'table')}")
print(f"- Visuals: {sum(1 for d in langchain_documents if d.metadata['type'] == 'chart_or_figure')}")
print(f"- Text: {sum(1 for d in langchain_documents if d.metadata['type'] == 'text')}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2721 > 512). Running this sequence through the model will result in indexing errors


Total LangChain documents indexed: 53
- Tables: 9
- Visuals: 2
- Text: 42


In [10]:
langchain_documents

[Document(metadata={'chunk_id': 'table_1_rows_1_4', 'type': 'table', 'page_number': 7, 'rows_range': '1-4', 'total_rows': 4, 'columns': ['System', 'LLMAgent', 'Similarity', 'Accuracy @0.85', 'Accuracy @0.9', 'Accuracy @0.95']}, page_content='### Table 1 (Page 7)\n\n| System   | LLMAgent      |   Similarity |   Accuracy @0.85 |   Accuracy @0.9 |   Accuracy @0.95 |\n|:---------|:--------------|-------------:|-----------------:|----------------:|-----------------:|\n| Baseline | GPT-3.5-turbo |       0.8639 |           0.5889 |          0.3667 |           0.06   |\n| Baseline | GPT-4o        |       0.8647 |           0.6444 |          0.4    |           0.1111 |\n| PIER-QA  | GPT-3.5-turbo |       0.8666 |           0.764  |          0.3708 |           0.1124 |\n| PIER-QA  | GPT-4o        |       0.8837 |           0.7191 |          0.4944 |           0.191  |'),
 Document(metadata={'chunk_id': 'table_2_rows_1_4', 'type': 'table', 'page_number': 7, 'rows_range': '1-4', 'total_rows': 4, '

In [11]:
len(langchain_documents)

53

## Vectordatabase Creation

In [13]:
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode
from qdrant_client import QdrantClient, models
from qdrant_client.http.models import Distance, SparseVectorParams, VectorParams

sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

# Create a Qdrant client for local storage
client = QdrantClient(path="./langchain_qdrant")

# Create a collection with both dense and sparse vectors
client.create_collection(
    collection_name="pdf_collection",
    vectors_config={"dense": VectorParams(size=1024, distance=Distance.COSINE)},
    sparse_vectors_config={
        "sparse": SparseVectorParams(index=models.SparseIndexParams(on_disk=False))
    },
)

qdrant = QdrantVectorStore(
    client=client,
    collection_name="pdf_collection",
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="dense",
    sparse_vector_name="sparse",
)

qdrant.add_documents(langchain_documents)
print(f"Indexed {len(langchain_documents)} chunks.")

Fetching 18 files: 100%|██████████| 18/18 [00:00<00:00, 23.09it/s]


Indexed 53 chunks.
